# ML-09 — Validation and Research Claim Audit

## 1. Two paper findings + my methodology questions

Both questions are asked in the paper's own spirit — it treats external beliefs as *hypotheses, not proof* and labels findings observational. My questions are about **label provenance** and **validation design**, not whether the analysts were careful.

**Finding #2 — "The Content Performance Curve" (peak at 61–90 days, decay cliff at 271–365).**
- *Where does the label come from?* The outcome is the **Health Score**, a FlyRank composite (impressions + position + CTR + scroll). It is a constructed index, not an observed traffic outcome, so "decay" is decay *of the index* — a page can move purely because one component shifts. A raw single outcome (impressions or clicks) would carry the claim more cleanly.
- *Does the design carry the claim?* The curve is **cross-sectional** (different pages at different ages at one snapshot) yet read as a **lifecycle**. That is a longitudinal claim a cross-section cannot support, because of **survivorship**: pages that survived to 365+ days are a selected set. The paper hedges this well ("not evidence that age naturally reverses").

**Finding #4 — "The Freshness Multiplier" (refreshed 365+ pages show 3.2× health, 57× impressions).**
- *Where does the label come from?* Health Score + impressions, compared between **refreshed** and **un-refreshed** mature pages — an **observational, non-random** contrast. Teams choose which pages to refresh, so the groups differ before any refresh (selection bias), and the arrow can point backwards: pages already recovering are the ones people bother to refresh.
- *Does the design carry the claim?* Partly. "Refresh timing is one of the strongest **levers**" reads causal, but a lever implies intervention and this is a between-groups correlation. The paper is admirably transparent that the headline `361+` ratio of **283:1 rests on a single declining page** — an n = 1 denominator that should carry no weight.

**Why this matters to my lane — my numbers now corroborate the paper's *direction*.** Using a **longitudinal** forward label (each page's own April impressions vs its March window) I see the same shapes: decline is **non-monotonic in age** (mid-life pages churn most, echoing the peak-then-decay curve), and **recently-updated pages decline less** (0–30 days ≈ 0.35 vs 0.54 base, echoing the freshness lever). The corroboration strengthens the paper's practical advice — but my methodology questions stand: neither my correlation nor the paper's proves that *refreshing* a page *causes* the retention, only that fresher/mid-life pages *differ*. The cell below re-shows my numbers next to the paper's framing.

In [1]:
import pandas as pd, numpy as np
d = pd.read_csv("../outputs/feature_vector.csv")
BASE = d.future_decline.mean()
print(f"base future-decline rate: {BASE:.3f}\n")

d["age_q"] = pd.qcut(d.content_age_days, 4, labels=["youngest","Q2","Q3","oldest"], duplicates="drop")
print("MY longitudinal label - decline rate by content age (paper: peak-then-decay curve):")
print(d.groupby("age_q", observed=True)["future_decline"].agg(decline_rate="mean", n="count").round(3).to_string())

d["freshness"] = pd.cut(d.days_since_last_update, [-1,30,90,180,100000], labels=["0-30","31-90","91-180","181+"])
print("\nMY longitudinal label - decline rate by recency (paper: 'freshness is a strong lever'):")
print(d.groupby("freshness", observed=True)["future_decline"].agg(decline_rate="mean", n="count").round(3).to_string())
print("\n-> non-monotonic in age, and freshest pages decline least: same DIRECTION as the paper,")
print("   from a within-page forward label instead of a cross-sectional Health-Score snapshot.")

base future-decline rate: 0.544

MY longitudinal label - decline rate by content age (paper: peak-then-decay curve):
          decline_rate      n
age_q                        
youngest         0.459  37117
Q2               0.599  36608
Q3               0.636  35299
oldest           0.485  36006

MY longitudinal label - decline rate by recency (paper: 'freshness is a strong lever'):
           decline_rate      n
freshness                     
0-30              0.346  10423
31-90             0.551  45046
91-180            0.576  15554
181+              0.561  74007

-> non-monotonic in age, and freshest pages decline least: same DIRECTION as the paper,
   from a within-page forward label instead of a cross-sectional Health-Score snapshot.


## 2. My model under an honest split (before/after)

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

SEED = 42
NUM = ["log_impressions_prev_30d","log_clicks_prev_30d","log_sessions_prev_30d","log_impr_older_30d",
       "ctr_prev_30d","hist_impr_momentum","log_search_volume","competition","cpc",
       "word_count","char_count","content_age_days","days_since_last_update","age_tier_order",
       "has_keyword_data","has_word_count"]
CAT = ["content_type","main_intent","competition_level"]
y = d["future_decline"].to_numpy()

def pipe():
    return Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), NUM),
                                                ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
                     ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])

Xtr, Xte, ytr, yte = train_test_split(d[NUM+CAT], y, test_size=0.3, random_state=SEED)
pr = pipe().fit(Xtr, ytr).predict_proba(Xte)[:, 1]
print(f"BEFORE  random split : AUC={roc_auc_score(yte,pr):.3f}  base={yte.mean():.3f}")

tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=SEED).split(d, groups=d.client_id))
pg = pipe().fit(d.iloc[tr][NUM+CAT], y[tr]).predict_proba(d.iloc[te][NUM+CAT])[:, 1]
print(f"AFTER   grouped split: AUC={roc_auc_score(y[te],pg):.3f}  base={y[te].mean():.3f}")
print("\ngap in AUC = memorization the random split hid. Keep the grouped (AFTER) number.")

BEFORE  random split : AUC=0.653  base=0.542


AFTER   grouped split: AUC=0.557  base=0.388

gap in AUC = memorization the random split hid. Keep the grouped (AFTER) number.


**Before/after — the split is the whole story.** I re-run the Week-5 logistic model two ways on the same features and label. A **random** split lets rows from the *same client* land on both sides, so the model memorizes client character; a **grouped** split (hold out whole clients) asks the honest question.

The random split reports **AUC ≈ 0.65**; the grouped split reports **AUC ≈ 0.56**. That **~0.09 drop is the memorization** the random split hid — and the grouped number is the one I keep. Even the honest number is only a little above 0.5: on unseen clients, history-only features barely separate next-month decliners from holders. That weak-but-real signal is exactly why ML-08 concluded the transparent rule is the deployable artifact.

## 3. Leakage audit

In [3]:
tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=SEED).split(d, groups=d.client_id))
auc_honest = roc_auc_score(y[te], pipe().fit(d.iloc[tr][NUM+CAT], y[tr]).predict_proba(d.iloc[te][NUM+CAT])[:, 1])

leak = d.copy()
leak["label_leak"] = leak["future_decline"].astype(float)
NUM_L = NUM + ["label_leak"]
pl = Pipeline([("pre", ColumnTransformer([("num", StandardScaler(), NUM_L),
                                          ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])),
               ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
auc_leak = roc_auc_score(y[te], pl.fit(leak.iloc[tr][NUM_L+CAT], y[tr]).predict_proba(leak.iloc[te][NUM_L+CAT])[:, 1])
print(f"honest AUC (final history-only features): {auc_honest:.3f}")
print(f"leaky  AUC (+ label-derived column):      {auc_leak:.3f}   <- jumps toward 1.0: harness works")

FORBIDDEN = {"impr_future","label_leak","future_decline","gsc_avg_position","gsc_sum_position",
             "health_score","content_updated_date","provider_used","model_used","content_id","client_id"}
model_cols = set(NUM + CAT)
assert not (FORBIDDEN & model_cols), FORBIDDEN & model_cols
print("\nguard passed: no forbidden / label-derived / product-flag column in the model's features.")

honest AUC (final history-only features): 0.557
leaky  AUC (+ label-derived column):      1.000   <- jumps toward 1.0: harness works

guard passed: no forbidden / label-derived / product-flag column in the model's features.


**Leakage audit on the final feature set — still clean.** I repeat the Week-3 hunt on the exact history-only vector the model ships with. Adding a **label-derived column** (here a copy of `future_decline`, standing in for the April label window itself) sends grouped-split AUC toward **~1.0** — the harness still detects a leak — and removing it returns the honest **~0.56**. The programmatic guard re-confirms none of the forbidden columns (the April label window, `gsc_avg_position` over the label window, the Health Score / product flags, identifiers) is present. No future window, no label-derived column, no decision-flag feeds the model.

## 4. Claim rewrite

**My boldest sentence, rewritten.**

> ❌ *Before (overclaims):* "My model predicts which pages will decline and tells the team what to fix."

> ✅ *After (safe):* "On held-out **unseen clients**, history-only signals separate next-month decliners from holders only **weakly** (grouped-split **observed** AUC ≈ 0.56; base rate 0.54), and a learned model **did not beat** a transparent three-condition rule (rule precision@50 ≈ 0.66 vs model ≈ 0.60 on reach-relevant pages). The rule is offered as **directional, decision-support** triage — *where an editor looks first* on the March-2026 anchor slice. It is **not** a prediction of Google's algorithm, **not** a causal claim that any page will drop, and **not** a promise that refreshing a flagged page reverses a decline."

The rewrite reports the honest grouped number and the base rate, states plainly that the model did not beat the rule, and names the three things the work can never claim.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.